# GRAPE + Low-Dimensional CMA-ES

This notebook uses the existing cavity-qubit Hamiltonian model to:

1. run GRAPE and get a center pulse `u0`,
2. generate a small number of smooth directions `V`,
3. run black-box CMA-ES on `u = u0 + V alpha`,
4. evaluate candidates with a fake experiment that can later be replaced by hardware.

In [ ]:
import importlib
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import hybrid_grape.cmaes as cmaes_module
importlib.reload(cmaes_module)

from hybrid_grape import (
    CalibrationConfig,
    FockPhysicsModel,
    GrapeConfig,
    PhysicsParams,
    SimulationConfig,
    fit_parametric_calibration,
    make_calibrated_model,
    optimize_physics_grape,
    sample_binomial_measurements,
)
from hybrid_grape.cmaes import (
    LowDimCMAESConfig,
    make_smooth_directions,
    optimize_lowdim_cma_es,
    pulses_from_alpha,
)
from hybrid_grape.config import khz_to_rad_per_us

jax.config.update("jax_enable_x64", False)


## Model Setup

`physics_model` is the Hamiltonian model used by GRAPE. `true_model` is only the fake experiment. On hardware, keep `physics_model` and replace the fake measurement function.

In [ ]:
seed = 1234
key = jax.random.key(seed)

q = SimulationConfig(
    n_cav=25,
    target_n=2,
    initial_cavity_n=0,
    initial_qubit_state=0,
    t_drive=1.408,
    ndt_drive=80,
    num_coeffs=20,
    spline_degree=2,
    spline_skip_left=2,
    spline_skip_right=2,
    param_clip=2.0,
)

p1 = PhysicsParams()
physics_model = FockPhysicsModel(q, p1)

true_params = PhysicsParams(
    chi=p1.chi + khz_to_rad_per_us(18.0),
    cavity_self_kerr=p1.cavity_self_kerr + khz_to_rad_per_us(0.65),
    cavity_detuning=khz_to_rad_per_us(14.0),
    qubit_detuning=khz_to_rad_per_us(-18.0),
    mu_qub=p1.mu_qub * 1.080,
    mu_cav=p1.mu_cav * 0.900,
    cavity_phase=0.18,
)
experiment_model = FockPhysicsModel(q, true_params)
# Alias used only for hidden diagnostics in the fake experiment.
true_model = experiment_model

print("parameter size:", physics_model.parameter_size)
print("parameter shape:", physics_model.parameter_shape)
print("chi [rad/us]:", p1.chi)
print("self Kerr [rad/us]:", p1.cavity_self_kerr)

## Fake Experiment Hook

This is the only function that needs to change when moving from the fake experiment to OPX/hardware.

In [ ]:
def measure_on_experiment(controls, key, shots):
    return sample_binomial_measurements(
        experiment_model,
        controls,
        key,
        shots=shots,
    )

## Sanity Check: The Two Models Are Different

Before optimizing, check that the fake experiment model is actually different from the GRAPE physics model.

In [ ]:
key, sanity_key = jax.random.split(key)
sanity_controls = 0.25 * jax.random.normal(sanity_key, (24, physics_model.parameter_size))
sanity_controls = jnp.clip(sanity_controls, -q.param_clip, q.param_clip)

sanity_p_physics = physics_model.population_probability(sanity_controls)
sanity_p_hybrid = experiment_model.population_probability(sanity_controls)

print("mean |P_hybrid - P_physics|:", float(jnp.mean(jnp.abs(sanity_p_hybrid - sanity_p_physics))))
print("max  |P_hybrid - P_physics|:", float(jnp.max(jnp.abs(sanity_p_hybrid - sanity_p_physics))))

plt.figure(figsize=(5, 5))
plt.scatter(sanity_p_physics, sanity_p_hybrid, alpha=0.8)
plt.plot([0, 1], [0, 1], color="black", linewidth=1)
plt.xlabel("P_physics: GRAPE model")
plt.ylabel("P_hybrid: fake experiment model")
plt.title("Sanity check: model mismatch")
plt.grid(True, alpha=0.3)
plt.show()


## Run GRAPE Center Pulse

This uses a differentiable GRAPE optimizer. It optimizes only `P_physics(u)` from the Hamiltonian model.

In [ ]:
key, init_key = jax.random.split(key)
initial_controls = 0.02 * jax.random.normal(init_key, (physics_model.parameter_size,))
initial_controls = jnp.clip(initial_controls, -q.param_clip, q.param_clip)

grape_config = GrapeConfig(
    maxiter=80,
    noise_samples=4,
    control_noise_std=0.0,
    amplitude_l2=1e-4,
    smoothness_l2=1e-4,
    param_clip=q.param_clip,
    grad_clip_norm=20.0,
)

grape_controls, grape_history, grape_summary, key = optimize_physics_grape(
    physics_model,
    initial_controls,
    key,
    grape_config,
)

print("GRAPE physics probability:", float(physics_model.photon_probability(grape_controls)))
print("GRAPE hidden true probability:", float(true_model.photon_probability(grape_controls)))
print("GRAPE summary columns: objective, P_physics, regularization_penalty, noisy_mean_P_physics, noisy_std_P_physics")
print("GRAPE summary:", grape_summary)

## Build Smooth Low-Dimensional Search Directions

The search is over `alpha`, not the full 80-dimensional pulse. The directions live in B-spline coefficient space and are smoothed channel-wise.

In [ ]:
num_directions = 8
direction_rms = 0.035

key, direction_key = jax.random.split(key)
directions = make_smooth_directions(
    direction_key,
    physics_model.parameter_shape,
    num_directions=num_directions,
    direction_rms=direction_rms,
)

print("directions shape:", directions.shape)
print("direction RMS:", jnp.sqrt(jnp.mean(directions**2, axis=1)))

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
channel_names = ["qubit I", "qubit Q", "cavity I", "cavity Q"]
center = grape_controls.reshape(4, -1)
for channel, ax in enumerate(axes):
    ax.plot(center[channel], color="black", linewidth=2, label="GRAPE center")
    for idx in range(min(4, num_directions)):
        ax.plot(center[channel] + directions[idx].reshape(4, -1)[channel], alpha=0.55)
    ax.set_ylabel(channel_names[channel])
    ax.grid(True, alpha=0.3)
axes[0].legend()
axes[-1].set_xlabel("B-spline coefficient index")
plt.show()

## Run Low-Dimensional CMA-ES

This evaluates batches of candidates with binomial measurements from the fake experiment. For hardware, the same optimizer can call the real experiment function.

In [ ]:
cma_config = LowDimCMAESConfig(
    num_directions=num_directions,
    generations=18,
    population_size=None,
    sigma0=0.40,
    alpha_clip=2.0,
    direction_rms=direction_rms,
    shots_per_candidate=250,
    param_clip=q.param_clip,
    seed=7,
)

cma_result, key = optimize_lowdim_cma_es(
    grape_controls,
    directions,
    measure_on_experiment,
    key,
    cma_config,
)

print("CMA-ES best measured probability:", cma_result.best_measured)
print("CMA-ES hidden true probability:", float(true_model.photon_probability(cma_result.controls)))
print("CMA-ES physics probability:", float(physics_model.photon_probability(cma_result.controls)))
print("best alpha:", cma_result.alpha)

## Diagnostics

## Method Metrics

This section makes the optimization bookkeeping explicit:

- GRAPE optimizes `P_physics(u)` from the Hamiltonian model.
- CMA-ES optimizes `P_exp = successes / shots`.
- In the fake experiment, shots are sampled from a second mismatched model `P_hybrid(u)`.
- On hardware, `P_hybrid` is not known; only `P_exp` is available.

In [ ]:
def probability_report(label, controls):
    p_physics = float(physics_model.photon_probability(controls))
    p_true = float(true_model.photon_probability(controls))
    return {
        "label": label,
        "physics_probability": p_physics,
        "fake_true_probability": p_true,
        "physics_error_vs_fake_true": p_true - p_physics,
    }


reports = [
    probability_report("initial", initial_controls),
    probability_report("GRAPE center", grape_controls),
    probability_report("CMA-ES best", cma_result.controls),
]

for row in reports:
    print(row)

print("\nWhat GRAPE optimized: physics_model.photon_probability(controls)")
print("What CMA-ES optimized: measured successes / shots from measure_on_experiment")
print("CMA-ES search dimension:", int(cma_result.directions.shape[0]))
print("Full pulse parameter dimension:", int(physics_model.parameter_size))

In [ ]:
if not hasattr(cma_result, "candidate_history"):
    raise RuntimeError(
        "This cma_result was created before candidate_history was added. "
        "Restart the kernel, run the import cell, then rerun the CMA-ES cell."
    )
candidate_history = cma_result.candidate_history
num_alpha_columns = candidate_history.shape[1] - 7
candidate_columns = (
    ["generation", "candidate", "measured_probability", "successes", "shots", "fake_true_probability", "rank"]
    + [f"alpha_{i}" for i in range(num_alpha_columns)]
)

print("candidate history shape:", candidate_history.shape)
print("columns:")
for idx, name in enumerate(candidate_columns):
    print(idx, name)

total_candidate_evaluations = int(candidate_history.shape[0])
total_shots = int(jnp.sum(candidate_history[:, 4]))
best_candidate_index = int(jnp.argmax(candidate_history[:, 2]))

print("\nTotal CMA-ES candidate evaluations:", total_candidate_evaluations)
print("Total CMA-ES shots:", total_shots)
print("Best measured candidate row:")
print(dict(zip(candidate_columns[:7], map(float, candidate_history[best_candidate_index, :7]))))

In [ ]:
history = cma_result.history
generation = history[:, 0]

shots_by_generation = []
successes_by_generation = []
measured_by_generation = []

for gen in generation.astype(int):
    mask = candidate_history[:, 0] == gen
    gen_successes = jnp.sum(candidate_history[mask, 3])
    gen_shots = jnp.sum(candidate_history[mask, 4])
    shots_by_generation.append(gen_shots)
    successes_by_generation.append(gen_successes)
    measured_by_generation.append(gen_successes / gen_shots)

shots_by_generation = jnp.asarray(shots_by_generation)
successes_by_generation = jnp.asarray(successes_by_generation)
measured_by_generation = jnp.asarray(measured_by_generation)
cumulative_shots = jnp.cumsum(shots_by_generation)
cumulative_successes = jnp.cumsum(successes_by_generation)
cumulative_measured = cumulative_successes / cumulative_shots

print("shots per generation:", shots_by_generation)
print("cumulative shots:", cumulative_shots)
print("cumulative measured success fraction:", cumulative_measured)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

axes[0].plot(history[:, 0], history[:, 1], marker="o", label="best measured in generation")
axes[0].plot(history[:, 0], history[:, 2], marker="o", label="mean measured in generation")
axes[0].plot(history[:, 0], history[:, 3], marker="o", label="best measured so far")
axes[0].plot(history[:, 0], history[:, 4], marker="o", label="hidden true of generation best", alpha=0.7)
axes[0].set_ylabel("target probability")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].bar(history[:, 0], shots_by_generation, label="shots this generation")
axes[1].plot(history[:, 0], cumulative_shots, color="tab:red", marker="o", label="cumulative shots")
axes[1].set_ylabel("shots")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(history[:, 0], history[:, 5], marker="o", label="CMA sigma")
axes[2].set_xlabel("CMA-ES generation")
axes[2].set_ylabel("sigma")
axes[2].grid(True, alpha=0.3)
axes[2].legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(
    candidate_history[:, 4],
    candidate_history[:, 2],
    c=candidate_history[:, 0],
    cmap="viridis",
    alpha=0.8,
)
plt.colorbar(label="generation")
plt.xlabel("shots per candidate")
plt.ylabel("measured successes / shots")
plt.title("Every CMA-ES candidate measurement")
plt.grid(True, alpha=0.3)
plt.show()

## Full Metrics Dashboard

These plots correspond to the metrics in your checklist for the GRAPE + CMA-ES version:

- `P_physics` means the Hamiltonian model optimized by GRAPE.
- `P_hybrid` means the hidden mismatched model used by the fake experiment.
- `P_exp` means the measured shot fraction `successes / shots`.
- support is a local CMA-ES trust proxy in alpha-space.
- mismatch magnitude means `abs(P_hybrid - P_physics)` or `abs(P_exp - P_physics)` for tested pulses.

In [ ]:
# Rebuild every tested CMA-ES pulse from the stored alpha history.
history = cma_result.history
if not hasattr(cma_result, "candidate_history"):
    raise RuntimeError(
        "This cma_result was created before candidate_history was added. "
        "Restart the kernel, run the import cell, then rerun the CMA-ES cell."
    )
candidate_history = cma_result.candidate_history
alpha_history = candidate_history[:, 7:]
candidate_controls = pulses_from_alpha(
    grape_controls,
    directions,
    alpha_history,
    param_clip=q.param_clip,
)

generation = candidate_history[:, 0].astype(int)
p_exp = candidate_history[:, 2]
successes = candidate_history[:, 3]
shots = candidate_history[:, 4]

# Two-model comparison:
# P_physics is the model optimized by GRAPE.
# P_hybrid is the mismatched hidden model used by the fake experiment.
p_physics = physics_model.population_probability(candidate_controls)
p_hybrid = experiment_model.population_probability(candidate_controls)
p_pred = p_physics

shot_noise = p_exp - p_hybrid
model_mismatch = p_hybrid - p_physics
measured_model_mismatch = p_exp - p_physics
measured_mismatch = measured_model_mismatch
mismatch_magnitude = jnp.abs(model_mismatch)
measured_mismatch_magnitude = jnp.abs(measured_model_mismatch)

model_mse = jnp.mean((p_physics - p_exp) ** 2)
weighted_model_mse = jnp.sum(shots * (p_physics - p_exp) ** 2) / jnp.sum(shots)
hidden_mismatch_mse = jnp.mean((p_physics - p_hybrid) ** 2)

unique_generations = jnp.unique(generation)
best_exp_by_generation = []
mean_exp_by_generation = []
best_physics_by_generation = []
best_hybrid_by_generation = []
mse_by_generation = []
weighted_mse_by_generation = []
hidden_mismatch_by_generation = []
shots_by_generation = []
best_seen_by_generation = []
baseline_physics = float(physics_model.photon_probability(grape_controls))
baseline_hybrid = float(experiment_model.photon_probability(grape_controls))

best_seen = -1.0
for gen in unique_generations:
    mask = generation == gen
    gen_exp = p_exp[mask]
    gen_physics = p_physics[mask]
    gen_hybrid = p_hybrid[mask]
    gen_shots = shots[mask]
    best_seen = max(best_seen, float(jnp.max(gen_exp)))
    best_exp_by_generation.append(jnp.max(gen_exp))
    mean_exp_by_generation.append(jnp.mean(gen_exp))
    best_physics_by_generation.append(jnp.max(gen_physics))
    best_hybrid_by_generation.append(jnp.max(gen_hybrid))
    mse_by_generation.append(jnp.mean((gen_physics - gen_exp) ** 2))
    weighted_mse_by_generation.append(jnp.sum(gen_shots * (gen_physics - gen_exp) ** 2) / jnp.sum(gen_shots))
    hidden_mismatch_by_generation.append(jnp.mean((gen_hybrid - gen_physics) ** 2))
    shots_by_generation.append(jnp.sum(gen_shots))
    best_seen_by_generation.append(best_seen)

best_exp_by_generation = jnp.asarray(best_exp_by_generation)
mean_exp_by_generation = jnp.asarray(mean_exp_by_generation)
best_physics_by_generation = jnp.asarray(best_physics_by_generation)
best_hybrid_by_generation = jnp.asarray(best_hybrid_by_generation)
mse_by_generation = jnp.asarray(mse_by_generation)
weighted_mse_by_generation = jnp.asarray(weighted_mse_by_generation)
hidden_mismatch_by_generation = jnp.asarray(hidden_mismatch_by_generation)
shots_by_generation = jnp.asarray(shots_by_generation)
best_seen_by_generation = jnp.asarray(best_seen_by_generation)
cumulative_shots = jnp.cumsum(shots_by_generation)

alpha_norm = jnp.linalg.norm(alpha_history, axis=1)
support_alpha = jnp.exp(-0.5 * (alpha_norm / cma_config.alpha_clip) ** 2)

print("Total tested CMA-ES pulses:", int(candidate_controls.shape[0]))
print("Total CMA-ES shots:", int(jnp.sum(shots)))
print("GRAPE baseline P_physics:", baseline_physics)
print("GRAPE baseline P_hybrid / fake experiment model:", baseline_hybrid)
print("Best measured CMA-ES P_exp:", float(jnp.max(p_exp)))
print("Best hidden mismatched model P_hybrid:", float(jnp.max(p_hybrid)))
print("Physics-vs-measured MSE:", float(model_mse))
print("Physics-vs-measured weighted MSE:", float(weighted_model_mse))
print("Hidden model mismatch MSE, P_physics vs P_hybrid:", float(hidden_mismatch_mse))
print("Best measured improvement over GRAPE fake-experiment baseline:", float(jnp.max(p_exp) - baseline_hybrid))


### 1. Final Goal Metric: Experimental Performance

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].plot(unique_generations, best_seen_by_generation, marker="o", label="best measured pulse so far")
axes[0].plot(unique_generations, best_exp_by_generation, marker="o", label="best measured in generation")
axes[0].plot(unique_generations, mean_exp_by_generation, marker="o", label="average tested pulse")
axes[0].axhline(baseline_physics, color="black", linestyle="--", label="GRAPE physics baseline")
axes[0].set_ylabel("P_exp = successes / shots")
axes[0].set_title("Experimental performance over CMA-ES generations")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(unique_generations, best_seen_by_generation - baseline_physics, marker="o", color="tab:green")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set_xlabel("CMA-ES generation")
axes[1].set_ylabel("best P_exp - GRAPE baseline")
axes[1].set_title("Improvement over GRAPE baseline")
axes[1].grid(True, alpha=0.3)
plt.show()

### 2. Model Quality Metrics: Prediction vs Measurement

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sc = axes[0].scatter(p_pred, p_exp, c=generation, cmap="viridis", alpha=0.8)
axes[0].plot([0, 1], [0, 1], color="black", linewidth=1)
axes[0].set_xlabel("P_physics")
axes[0].set_ylabel("P_exp = successes / shots")
axes[0].set_title("Predicted vs measured")
axes[0].grid(True, alpha=0.3)
fig.colorbar(sc, ax=axes[0], label="generation")

axes[1].plot(unique_generations, mse_by_generation, marker="o", label="MSE")
axes[1].plot(unique_generations, weighted_mse_by_generation, marker="o", label="weighted MSE")
axes[1].set_xlabel("generation")
axes[1].set_ylabel("model error")
axes[1].set_title("Physics vs fake experiment error")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].scatter(p_physics, measured_mismatch, c=generation, cmap="viridis", alpha=0.8)
axes[2].axhline(0.0, color="black", linewidth=1)
axes[2].set_xlabel("P_physics")
axes[2].set_ylabel("P_exp - P_physics")
axes[2].set_title("Measured mismatch over GRAPE physics model")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Two-Model Mimic of an Experiment

This is the key comparison: GRAPE optimized `P_physics`, while CMA-ES receives samples from the mismatched fake experiment `P_hybrid`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(p_physics, p_hybrid, c=generation, cmap="viridis", alpha=0.8)
axes[0].plot([0, 1], [0, 1], color="black", linewidth=1)
axes[0].set_xlabel("P_physics: GRAPE model")
axes[0].set_ylabel("P_hybrid: mismatched fake experiment model")
axes[0].set_title("Model mismatch seen by experiment")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(p_hybrid, p_exp, c=generation, cmap="viridis", alpha=0.8)
axes[1].plot([0, 1], [0, 1], color="black", linewidth=1)
axes[1].set_xlabel("P_hybrid")
axes[1].set_ylabel("P_exp = successes / shots")
axes[1].set_title("Shot noise around experiment model")
axes[1].grid(True, alpha=0.3)

axes[2].scatter(p_physics, p_exp, c=generation, cmap="viridis", alpha=0.8)
axes[2].plot([0, 1], [0, 1], color="black", linewidth=1)
axes[2].set_xlabel("P_physics")
axes[2].set_ylabel("P_exp")
axes[2].set_title("What CMA-ES actually sees vs GRAPE model")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Parametric Calibration

Instead of learning a black-box correction over the full pulse, fit a small set of physical mismatch parameters from the measured pulses. Here the fit uses the CMA-ES candidate dataset `(u_k, P_exp(u_k))` and solves for corrections to chi, Kerr, detunings, drive scales, and cavity phase.


In [ ]:
calibration_fit = fit_parametric_calibration(
    q,
    p1,
    candidate_controls,
    p_exp,
    shots,
    initial=CalibrationConfig(),
)
calibrated_model = make_calibrated_model(q, p1, calibration_fit.config)
p_calibrated = calibrated_model.population_probability(candidate_controls)

print("fit success:", calibration_fit.success)
print("fit message:", calibration_fit.message)
print("weighted fit loss:", calibration_fit.loss)
print("fitted calibration parameters:")
print(calibration_fit.config)

mse_physics_to_exp = jnp.mean((p_physics - p_exp) ** 2)
mse_calibrated_to_exp = jnp.mean((p_calibrated - p_exp) ** 2)
mse_physics_to_hybrid = jnp.mean((p_physics - p_hybrid) ** 2)
mse_calibrated_to_hybrid = jnp.mean((p_calibrated - p_hybrid) ** 2)

print("MSE P_physics vs P_exp:", float(mse_physics_to_exp))
print("MSE P_calibrated vs P_exp:", float(mse_calibrated_to_exp))
print("MSE P_physics vs hidden P_hybrid:", float(mse_physics_to_hybrid))
print("MSE P_calibrated vs hidden P_hybrid:", float(mse_calibrated_to_hybrid))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(p_physics, p_exp, c=generation, cmap="viridis", alpha=0.8)
axes[0].plot([0, 1], [0, 1], color="black", linewidth=1)
axes[0].set_xlabel("P_physics")
axes[0].set_ylabel("P_exp")
axes[0].set_title("Before calibration")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(p_calibrated, p_exp, c=generation, cmap="viridis", alpha=0.8)
axes[1].plot([0, 1], [0, 1], color="black", linewidth=1)
axes[1].set_xlabel("P_calibrated")
axes[1].set_ylabel("P_exp")
axes[1].set_title("After parametric calibration")
axes[1].grid(True, alpha=0.3)

axes[2].scatter(p_hybrid, p_calibrated, c=generation, cmap="viridis", alpha=0.8)
axes[2].plot([0, 1], [0, 1], color="black", linewidth=1)
axes[2].set_xlabel("hidden P_hybrid")
axes[2].set_ylabel("P_calibrated")
axes[2].set_title("Calibrated model vs hidden model")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Calibrated GRAPE

Now rerun GRAPE using the calibrated physical model. In the real experiment this would produce the next pulse to test on hardware.


In [ ]:
calibrated_grape_controls, calibrated_grape_history, calibrated_grape_summary, key = optimize_physics_grape(
    calibrated_model,
    grape_controls,
    key,
    grape_config,
)

print("Original GRAPE P_physics:", float(physics_model.photon_probability(grape_controls)))
print("Original GRAPE hidden P_hybrid:", float(experiment_model.photon_probability(grape_controls)))
print("CMA-ES best measured P_exp:", cma_result.best_measured)
print("CMA-ES best hidden P_hybrid:", float(experiment_model.photon_probability(cma_result.controls)))
print("Calibrated GRAPE P_calibrated:", float(calibrated_model.photon_probability(calibrated_grape_controls)))
print("Calibrated GRAPE hidden P_hybrid:", float(experiment_model.photon_probability(calibrated_grape_controls)))
print("Calibrated GRAPE original P_physics:", float(physics_model.photon_probability(calibrated_grape_controls)))


In [ ]:
labels = ["GRAPE", "CMA-ES", "calibrated GRAPE"]
hidden_scores = jnp.array([
    experiment_model.photon_probability(grape_controls),
    experiment_model.photon_probability(cma_result.controls),
    experiment_model.photon_probability(calibrated_grape_controls),
])
physics_scores = jnp.array([
    physics_model.photon_probability(grape_controls),
    physics_model.photon_probability(cma_result.controls),
    physics_model.photon_probability(calibrated_grape_controls),
])
calibrated_scores = jnp.array([
    calibrated_model.photon_probability(grape_controls),
    calibrated_model.photon_probability(cma_result.controls),
    calibrated_model.photon_probability(calibrated_grape_controls),
])

x = jnp.arange(len(labels))
width = 0.25
plt.figure(figsize=(9, 4))
plt.bar(x - width, physics_scores, width, label="P_physics")
plt.bar(x, calibrated_scores, width, label="P_calibrated")
plt.bar(x + width, hidden_scores, width, label="hidden P_hybrid")
plt.xticks(x, labels)
plt.ylabel("target probability")
plt.title("Pulse comparison after parametric calibration")
plt.grid(True, axis="y", alpha=0.3)
plt.legend()
plt.show()


### 3. Trust / Extrapolation Metrics

For this CMA-ES version, trust is measured in the low-dimensional alpha space. Large `||alpha||` means the pulse is farther from the GRAPE center. `support_alpha` is a simple locality score near the GRAPE center.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(alpha_norm, p_exp, c=generation, cmap="viridis", alpha=0.8)
axes[0].set_xlabel("||alpha||")
axes[0].set_ylabel("P_exp")
axes[0].set_title("Performance vs distance from GRAPE pulse")
axes[0].grid(True, alpha=0.3)

axes[1].hist(support_alpha, bins=20, color="tab:blue", alpha=0.75)
axes[1].set_xlabel("local alpha support proxy")
axes[1].set_ylabel("count")
axes[1].set_title("Support distribution")
axes[1].grid(True, alpha=0.3)

axes[2].scatter(support_alpha, mismatch_magnitude, c=generation, cmap="viridis", alpha=0.8)
axes[2].set_xlabel("local alpha support proxy")
axes[2].set_ylabel("|P_hybrid - P_physics|")
axes[2].set_title("Mismatch magnitude vs support")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_row = int(jnp.argmax(p_exp))
print("Best pulse alpha norm:", float(alpha_norm[best_row]))
print("Best pulse support proxy:", float(support_alpha[best_row]))
print("Best pulse mismatch magnitude |P_hybrid - P_physics|:", float(mismatch_magnitude[best_row]))

### 4. Optimization Metrics: GRAPE and CMA-ES Behavior

In [ ]:
# GRAPE history columns from optimize_physics_grape:
# objective, P_physics, regularization_penalty, noisy_mean_P_physics, noisy_std_P_physics
grape_cols = {
    "objective": 0,
    "physics_probability": 1,
    "penalty": 2,
    "noisy_mean": 3,
    "noisy_std": 4,
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(grape_history[:, grape_cols["objective"]], label="J(u)")
axes[0, 0].set_title("GRAPE objective")
axes[0, 0].set_xlabel("GRAPE iteration")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(grape_history[:, grape_cols["physics_probability"]], label="P_physics")
axes[0, 1].plot(grape_history[:, grape_cols["noisy_mean"]], label="mean noisy P_physics", linestyle="--")
axes[0, 1].set_title("GRAPE physics probability")
axes[0, 1].set_xlabel("GRAPE iteration")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].plot(grape_history[:, grape_cols["penalty"]], label="regularization penalty")
axes[1, 0].set_title("Pulse regularization")
axes[1, 0].set_xlabel("GRAPE iteration")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(unique_generations, history[:, 5], marker="o", label="CMA sigma")
axes[1, 1].plot(unique_generations, best_seen_by_generation, marker="o", label="CMA best measured")
axes[1, 1].set_title("CMA-ES behavior")
axes[1, 1].set_xlabel("CMA-ES generation")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()


### 5. Closed-Loop Performance Summary

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 10), sharex=True)

axes[0].plot(unique_generations, best_seen_by_generation, marker="o")
axes[0].set_ylabel("best P_exp")
axes[0].set_title("Best experimental pulse")
axes[0].grid(True, alpha=0.3)

axes[1].plot(unique_generations, cumulative_shots, marker="o", color="tab:orange")
axes[1].set_ylabel("dataset size / shots")
axes[1].set_title("Accumulated measurement budget")
axes[1].grid(True, alpha=0.3)

axes[2].plot(unique_generations, weighted_mse_by_generation, marker="o", color="tab:red")
axes[2].set_ylabel("weighted MSE")
axes[2].set_title("Model error per generation")
axes[2].grid(True, alpha=0.3)

support_best_by_generation = []
for gen in unique_generations:
    mask = generation == gen
    local_best = jnp.argmax(p_exp[mask])
    support_best_by_generation.append(support_alpha[mask][local_best])
support_best_by_generation = jnp.asarray(support_best_by_generation)

axes[3].plot(unique_generations, support_best_by_generation, marker="o", color="tab:green")
axes[3].set_xlabel("CMA-ES generation")
axes[3].set_ylabel("support proxy")
axes[3].set_title("Support of best pulse in each generation")
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Plot Optimized B-Spline Pulses

This section shows both the B-spline coefficients and the reconstructed pulse waveforms on the simulation time grid. The hardware waveform should usually come from the reconstructed time-domain values, not directly from the coefficient plot.

In [ ]:
def split_coefficients(controls):
    return jnp.asarray(controls).reshape(4, -1)


def reconstruct_time_pulses(controls):
    coeffs = split_coefficients(controls)
    return coeffs @ physics_model.bsplines_mids


channel_names = ["qubit I", "qubit Q", "cavity I", "cavity Q"]
time_us = physics_model.t_mids

grape_coeffs = split_coefficients(grape_controls)
cma_coeffs = split_coefficients(cma_result.controls)
grape_waveforms = reconstruct_time_pulses(grape_controls)
cma_waveforms = reconstruct_time_pulses(cma_result.controls)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)

for channel, ax in enumerate(axes):
    ax.plot(grape_coeffs[channel], color="black", linewidth=2, label="GRAPE center")
    ax.plot(cma_coeffs[channel], color="tab:red", linewidth=1.8, label="CMA-ES best")
    ax.set_ylabel(channel_names[channel])
    ax.grid(True, alpha=0.3)

axes[0].legend()
axes[-1].set_xlabel("B-spline coefficient index")
fig.suptitle("Optimized B-spline coefficients", y=0.995)
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)

for channel, ax in enumerate(axes):
    ax.plot(time_us, grape_waveforms[channel], color="black", linewidth=2, label="GRAPE center")
    ax.plot(time_us, cma_waveforms[channel], color="tab:red", linewidth=1.8, label="CMA-ES best")
    ax.set_ylabel(channel_names[channel])
    ax.grid(True, alpha=0.3)

axes[0].legend()
axes[-1].set_xlabel("time [us]")
fig.suptitle("Reconstructed time-domain pulses from B-splines", y=0.995)
plt.show()

In [ ]:
history = cma_result.history
columns = ["generation", "batch_best", "batch_mean", "best_seen", "true_batch_best", "sigma"]
for i, name in enumerate(columns):
    print(i, name)

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(history[:, 0], history[:, 1], label="batch best measured")
axes[0].plot(history[:, 0], history[:, 2], label="batch mean measured")
axes[0].plot(history[:, 0], history[:, 3], label="best measured so far")
axes[0].plot(history[:, 0], history[:, 4], label="hidden true batch best", alpha=0.75)
axes[0].set_ylabel("P_n")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(history[:, 0], history[:, 5], label="CMA sigma")
axes[1].set_xlabel("generation")
axes[1].set_ylabel("sigma")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
u0 = grape_controls.reshape(4, -1)
u_best = cma_result.controls.reshape(4, -1)

for channel, ax in enumerate(axes):
    ax.plot(u0[channel], label="GRAPE center", color="black", linewidth=2)
    ax.plot(u_best[channel], label="CMA-ES best", color="tab:red", linewidth=1.8)
    ax.set_ylabel(channel_names[channel])
    ax.grid(True, alpha=0.3)
axes[0].legend()
axes[-1].set_xlabel("B-spline coefficient index")
plt.show()

## Optional: Inspect Any Alpha Candidate

Change `alpha_test` to look at another point in the local CMA-ES search space.

In [ ]:
alpha_test = cma_result.alpha
candidate = pulses_from_alpha(
    grape_controls,
    directions,
    alpha_test,
    param_clip=q.param_clip,
)

print("candidate physics probability:", float(physics_model.photon_probability(candidate)))
print("candidate hidden true probability:", float(true_model.photon_probability(candidate)))